AI Resume Screening System with Tracing


🔧  Install & Importing Libraries

In [1]:

!pip install -q langchain langchain-community langchain-openai transformers

In [2]:

import os

from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [3]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)

Job Description


In [4]:
job_description = """
Job Role: Data Scientist

Required Skills:
- Python
- Machine Learning
- Deep Learning
- SQL
- Data Visualization

Experience Required:
- Minimum 2+ years

Tools & Technologies:
- TensorFlow
- Pandas
- Scikit-learn
"""

Input Resumes

In [5]:
strong_resume = """
Candidate Name: Praveen

Skills:
- Python
- Machine Learning
- Deep Learning
- SQL

Tools:
- TensorFlow

Experience:
- 1 year

Projects:
- Worked on AI and Data Science projects
"""

average_resume = """
Candidate Name: Alice

Skills:
- Python
- SQL
- Basic Machine Learning

Experience:
- 1 year
"""

weak_resume = """
Candidate Name: Joe

Skills:
- HTML
- CSS
- Basic Excel

Experience:
- No relevant experience in Data Science
"""

Initializing Language Model

In [6]:
!pip uninstall -y transformers
!pip install transformers==4.41.2

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)


In [7]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_7210/2158257986.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


Step 1: Skill Extraction

In [8]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract the following from the resume.

Return ONLY in this exact format:

Skills: <comma separated>
Tools: <comma separated>
Experience: <number of years or text>

Resume:
{resume}
"""
)

def extract_skills(resume):
    return llm.invoke(extract_prompt.format(resume=resume)).strip()

Step 2:  Matching

In [9]:
match_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Compare the resume with the job description.

Return ONLY in this format:

Matching Skills: <comma separated>
Missing Skills: <comma separated>

Resume:
{resume}

Job Description:
{job}
"""
)

def match_skills(resume):
    return llm.invoke(match_prompt.format(resume=resume, job=job_description)).strip()

Step 3: Scoring

In [10]:
score_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Evaluate the resume.

Give:
- A score between 0 and 100
- A short reason

Return ONLY in this format:

Score: <number>
Reason: <text>

Resume:
{resume}

Job Description:
{job}
"""
)

def score_resume(resume):
    return llm.invoke(score_prompt.format(resume=resume, job=job_description)).strip()

Step 4
: Explanation

In [11]:
explain_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Explain the evaluation clearly.

Return in this format:

Strengths:
- point 1
- point 2

Weaknesses:
- point 1
- point 2

Final Justification:
<short paragraph>

Resume:
{resume}

Job Description:
{job}
"""
)

def explain_resume(resume):
    return llm.invoke(explain_prompt.format(resume=resume, job=job_description)).strip()

Pipeline

In [12]:
def run_pipeline(resume):
    print("===== EXTRACTION =====")
    print(extract_skills(resume))

    print("\n===== MATCHING =====")
    print(match_skills(resume))

    print("\n===== SCORE =====")
    print(score_resume(resume))

    print("\n===== EXPLANATION =====")
    print(explain_resume(resume))

Running the Pipeline

In [13]:
print("🔹 STRONG CANDIDATE")
run_pipeline(strong_resume)

print("\n🔹 AVERAGE CANDIDATE")
run_pipeline(average_resume)

print("\n🔹 WEAK CANDIDATE")
run_pipeline(weak_resume)

🔹 STRONG CANDIDATE
===== EXTRACTION =====
Praveen is a python programming language. He is proficient in deep learning and SQL.

===== MATCHING =====
Data Scientist

===== SCORE =====
0

===== EXPLANATION =====
Strengths: - point 1 - point 2 - point 3

🔹 AVERAGE CANDIDATE
===== EXTRACTION =====
Alice is a Python and SQL developer.

===== MATCHING =====
Data Scientist

===== SCORE =====
0

===== EXPLANATION =====
Strengths: - point 1 - point 2 - point 3 - point 4 - point 5 - point 6 - point 7 - point 8 - point 9 - point 10 - point 11 - point 12 - point 13 - point 14 - point 15 - point 16 - point 17 - point 18 - point 19 - point 20 - point 21 - point 22 - point 23 - point 24 - point 25 - point 26 - point 27 - point 28 - point 29 - point 30 - point 31 - point 32 - point 33 - point 34 - point 35 - point 36 - point 37 - point 38 - point 39 - point 40 - point 41 - point 42 - point 43 - point 42 - point 41 - point 42 - point 43 - point 41 - point 42 - point 43 - point 41 - point 42 - point 43 

Debugging

In [14]:
wrong_resume = "I am expert in everything"

run_pipeline(wrong_resume)


===== EXTRACTION =====
number of years or text>

===== MATCHING =====
Data Scientist

===== SCORE =====
0

===== EXPLANATION =====
Strengths: - point 1 - point 2 - point 3


✅Conclusion

This project demonstrates the practical implementation of an AI-powered resume screening system using modern LLM tools.

Key achievements include:

 • Built a modular LangChain pipeline for structured processing

 • Implemented skill extraction and matching without assumptions

 • Designed a transparent scoring system (0–100)

 • Enabled explainable AI outputs for better decision-making

 • Integrated LangSmith tracing for debugging and monitoring


The system simulates real-world AI recruitment tools, helping automate candidate evaluation while maintaining transparency and fairness.

⚠️ Limitations

 • Output quality depends heavily on prompt design

 • LLM may produce inconsistent formatting

 • No strict validation layer for outputs

🚀 Future Improvements

 • Add structured output parsing (JSON format)

 • Improve scoring with weighted criteria

 • Integrate real resume datasets

 • Enable UI for recruiter interaction

 • Add bias detection and fairness checks